# MCMC 采样系统

本教程演示 HIcosmo 的 MCMC 采样系统。

**关键 API**：
- `MCMC(params, likelihood)` — 创建 MCMC 采样器
- `mcmc.run(num_samples=N)` — 运行采样
- `mcmc.print_summary()` — 打印结果摘要
- `sampler='numpyro'|'emcee'` — 选择采样后端
- `information_criteria()` — 计算 AIC/BIC/Evidence

In [ ]:
import hicosmo as hc
hc.init()

## 1. 基本用法

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.models import LCDM
from hicosmo.likelihoods import SN_likelihood

# 创建似然
sne = SN_likelihood(LCDM, "pantheon+")

# 参数配置: (初始值, 最小值, 最大值)
params = {
    'H0': (70.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

# 创建并运行 MCMC
mcmc = MCMC(params, sne, chain_name='mcmc_demo')
samples = mcmc.run(num_samples=2000, num_warmup=500)
mcmc.print_summary()

## 2. 参数配置格式

In [ ]:
# 格式1: 元组 (init, min, max)
params_tuple = {'H0': (70.0, 60.0, 80.0)}

# 格式2: 字典 (更多控制)
params_dict = {
    'H0': {'init': 70.0, 'min': 60.0, 'max': 80.0, 'latex': r'$H_0$'}
}

# 格式3: 带先验分布
params_prior = {
    'H0': {'prior': {'dist': 'uniform', 'min': 60, 'max': 80}, 'ref': 70.0},
    'Omega_m': {'prior': {'dist': 'normal', 'loc': 0.3, 'scale': 0.05}, 'bounds': [0.1, 0.5]}
}

print("支持的先验: uniform, normal, truncated_normal")

## 3. NumPyro NUTS（默认）

基于梯度的 NUTS 采样器，适合标准宇宙学分析。

In [ ]:
# 显式指定 NumPyro
mcmc_nuts = MCMC(params, sne, chain_name='nuts_demo', sampler='numpyro')
samples = mcmc_nuts.run(num_samples=2000)
mcmc_nuts.print_summary()

## 4. emcee 集合采样器

无需梯度，适合复杂/多模态似然函数。

In [ ]:
mcmc_emcee = MCMC(params, sne, chain_name='emcee_demo', sampler='emcee')
samples = mcmc_emcee.run(num_samples=2000)
mcmc_emcee.print_summary()

## 5. 采样器选择指南

| 场景 | 推荐采样器 |
|------|------------|
| 标准分析 | NumPyro NUTS |
| 似然不稳定 | emcee |
| 高维 (>20 参数) | NumPyro NUTS |
| 多模态分布 | emcee |

## 6. 多核并行

In [ ]:
import jax
print(f"JAX 设备数: {jax.device_count()}")

# 多核配置（必须在 JAX 导入前）
# hc.init(8)  # 8 个并行设备

# chain_method 选项
print("chain_method: 'auto' (推荐), 'vectorized', 'sequential', 'parallel'")

## 7. 检查点和断点续跑

In [ ]:
# 启用检查点
mcmc_ckpt = MCMC(
    params, sne, chain_name='ckpt_demo',
    enable_checkpoints=True,
    checkpoint_interval=100
)

print("断点续跑: auto_resume=True")

## 8. 收敛诊断

In [ ]:
# Gelman-Rubin R̂ < 1.01 表示收敛
# ESS > 100 表示足够有效样本

mcmc_diag = MCMC(params, sne, chain_name='diag_demo')
samples = mcmc_diag.run(num_samples=4000)
mcmc_diag.print_summary()

## 9. 可视化

In [ ]:
from hicosmo.visualization import Plotter

plotter = Plotter('diag_demo')

# 角图
plotter.corner(['H0', 'Omega_m'], filename='figures/06_mcmc_corner.pdf')

# 链轨迹
plotter.traces(['H0', 'Omega_m'], filename='figures/06_mcmc_traces.pdf')

# 统计报告
plotter.report()

## 10. 信息判据 (AIC/BIC/Evidence)

$$\text{AIC} = \chi^2_{\min} + 2k, \quad \text{BIC} = \chi^2_{\min} + k \ln N$$

In [ ]:
from hicosmo.visualization import information_criteria

ic = information_criteria(
    samples=samples,
    log_likelihood_fn=sne,
    num_data=sne.n_sne,
    param_names=['H0', 'Omega_m'],
    max_samples=500
)

print(f"χ²_min = {ic['chi2_min']:.2f}")
print(f"AIC = {ic['aic']:.2f}")
print(f"BIC = {ic['bic']:.2f}")
print(f"ln(Evidence) = {ic['log_evidence']:.2f}")

## 11. 保存和加载

In [ ]:
# 保存结果
mcmc_diag.save_results('results/mcmc_demo.pkl')

# 加载结果
# loaded = MCMC.load_results('results/mcmc_demo.pkl')

## API 速查

```python
from hicosmo.samplers import MCMC
from hicosmo.visualization import Plotter, information_criteria

# 初始化
import hicosmo as hc
hc.init()  # 自动检测核数

# 参数配置
params = {
    'H0': (70.0, 60.0, 80.0),      # (init, min, max)
    'Omega_m': (0.3, 0.1, 0.5),
}

# 创建 MCMC
mcmc = MCMC(
    params,
    likelihood,
    chain_name='my_chain',
    sampler='numpyro',      # 或 'emcee'
    chain_method='auto',
    enable_checkpoints=True,
)

# 运行采样
samples = mcmc.run(
    num_samples=10000,
    num_warmup=1000,
    # num_chains 默认 = 设备数
)

# 查看结果
mcmc.print_summary()
mcmc.save_results('output.pkl')

# 可视化
plotter = Plotter('my_chain')
plotter.corner(['H0', 'Omega_m'])
plotter.traces(['H0', 'Omega_m'])
plotter.report()

# 信息判据
ic = information_criteria(
    samples=samples,
    log_likelihood_fn=likelihood,
    num_data=N,
    param_names=['H0', 'Omega_m']
)
print(f"AIC={ic['aic']:.2f}, BIC={ic['bic']:.2f}")
```